# Test notebook for DPD funcionality

In [1]:
import warnings

warnings.filterwarnings("ignore")

In [2]:
from flowermd.library import PPS

molecules = PPS(num_mols=50, lengths=30)
molecules.coarse_grain(beads={"_A": "c1cc(S)ccc1"})
molecules.bond_length

0.176

## New Random Walk System Class

In [3]:
from flowermd.library import RandomWalk
import unyt as u

ref_length = 0.3438 * u.Unit("nm")
ref_mass = 32.06 * u.Unit("amu")
ref_energy = 1.065 * u.Unit("kJ/mol")
ref_values_dict = {"length": ref_length, "mass": ref_mass, "energy": ref_energy}

system = RandomWalk(
    molecules=molecules,
    density=1.32 * u.Unit("g/cm**3"),
    bond_length=1.4226,
    buffer=0.58,
    base_units=ref_values_dict,
)

(1500, 3)


## FF from GMSO XML file

In [4]:
from flowermd.library.forcefields import Bead_Spring_DPD

In [5]:
system.apply_forcefield(force_field=Bead_Spring_DPD(), r_cut=1.15, kT=1.0)

## Forcefield class

from flowermd.library import DPD

A = 2000
gamma = 1500
kT = 1.0
r_cut = 1.4226
bond_k = 50000
bond_r0 = 1.4226
dpd_ff = DPD(
    A=A, gamma=gamma, kT=kT, r_cut=r_cut, bond_k=bond_k, bond_r0=bond_r0
)

In [6]:
from flowermd.library import PhantomWalk

A = 2000
gamma = 1500
kT = 1.0
r_cut = 1.4226
bond_k = 50000
bond_r0 = 1.4226
sim = PhantomWalk(
    initial_state=system.hoomd_snapshot,
    forcefield=system.hoomd_forcefield,
    gsd_write_freq=10,
    log_write_freq=50,
    n_steps_dpd=500,
    n_steps_fire=100,
)

Initializing simulation state from a gsd.hoomd.Frame.
Step 30 of 500; TPS: 312.44; ETA: 0.0 minutes
Step 60 of 500; TPS: 592.79; ETA: 0.0 minutes
Step 90 of 500; TPS: 849.07; ETA: 0.0 minutes
Step 120 of 500; TPS: 1076.31; ETA: 0.0 minutes
Step 150 of 500; TPS: 1281.72; ETA: 0.0 minutes
Step 180 of 500; TPS: 1478.33; ETA: 0.0 minutes
Step 210 of 500; TPS: 1644.34; ETA: 0.0 minutes
Step 240 of 500; TPS: 1812.14; ETA: 0.0 minutes
Step 270 of 500; TPS: 1963.37; ETA: 0.0 minutes
Step 300 of 500; TPS: 2103.09; ETA: 0.0 minutes
Step 330 of 500; TPS: 2240.46; ETA: 0.0 minutes
Step 360 of 500; TPS: 2359.96; ETA: 0.0 minutes
Step 390 of 500; TPS: 2471.89; ETA: 0.0 minutes
Step 420 of 500; TPS: 2578.16; ETA: 0.0 minutes
Step 450 of 500; TPS: 2679.54; ETA: 0.0 minutes
Step 480 of 500; TPS: 2781.77; ETA: 0.0 minutes
Step 10 of 100; TPS: 6157.64; ETA: 0.0 minutes
Step 40 of 100; TPS: 6258.8; ETA: 0.0 minutes
Step 70 of 100; TPS: 6247.77; ETA: 0.0 minutes


In [7]:
import hoomd

for writer in sim.operations.writers:
    if isinstance(writer, hoomd.write.GSD):
        writer.flush()

In [8]:
# system.to_gsd("random_walk_test.gsd")